In [1]:
import numpy as np
import pandas as pd


# ============================================================
# Configuration
# ============================================================

WINDOW_SIZE = 60


# ============================================================
# Selected process variables
# ============================================================

selected_variables = [

    # Temperature
    "T701", "T702", "T703", "T704", "T705",
    "T706", "T708", "T709", "T711", "T712",

    # Pressure
    "P701", "P702", "PDI701", "PDI702", "PY23",

    # Flow
    "FT703", "FT704", "FYI702",

    # Level
    "LS701", "LS702",

    # Heating / power
    "H002", "H701", "H702", "H704", "H706", "H708",

    # Vacuum
    "P301", "TV1",

    # Nitrogen
    "AV709",

    # Cooling
    "AV716",
]


# ============================================================
# Feature groups
# ============================================================

temperature_sensors = [
    "T701", "T702", "T703", "T704", "T705",
    "T706", "T708", "T709", "T711", "T712"
]

continuous_process_variables = [
    # Temperatures
    *temperature_sensors,

    # Pressure
    "P701", "P702", "PDI701", "PDI702", "PY23",

    # Flow
    "FT703", "FT704", "FYI702",

    # Level
    "LS701", "LS702",
]

actuator_variables = [
    # Heating / power
    "H002", "H701", "H702", "H704", "H706", "H708",

    # Vacuum
    "P301", "TV1",

    # Nitrogen
    "AV709",

    # Cooling
    "AV716",
]

# All variables used for generic window statistics and slopes
feature_variables = (
    continuous_process_variables
    + actuator_variables
)


# ============================================================
# Physically meaningful temperature differences
# ============================================================

difference_pairs = [
    ("T704", "T706"),
    ("T706", "T708"),
    ("T704", "T708"),
    ("T701", "T702"),
    ("T702", "T703"),
    ("T711", "T712"),
    ("T703", "T709"),
]


# ============================================================
# Metadata
# ============================================================

metadata_columns = [
    "identifier",
    "phase",
    "batch",
    "operating_point",
    "experiment_type",
    "experiment",
    "window",
    "window_start",
    "window_end",
    "anomaly_label",
]

feature_key = [
    "identifier",
    "window"
]


# ============================================================
# Load data
# ============================================================

operation_data = pd.read_parquet(
    "../data/processed/operation_data.parquet"
)


# ============================================================
# Validate required columns
# ============================================================

required_columns = (
    metadata_columns
    + selected_variables
)

missing_columns = [
    column
    for column in required_columns
    if column not in operation_data.columns
]

if missing_columns:
    raise ValueError(
        f"Missing columns: {missing_columns}"
    )


# ============================================================
# Basic dataset information
# ============================================================

print(f"Rows: {len(operation_data):,}")

print(
    f"Experiments: "
    f"{operation_data['identifier'].nunique():,}"
)

print(
    "Windows:",
    operation_data[feature_key]
    .drop_duplicates()
    .shape[0]
)


# ============================================================
# Keep only complete 60-second windows
# ============================================================

complete_windows = (
    operation_data
    .groupby(feature_key)
    .size()
    .loc[lambda x: x == WINDOW_SIZE]
    .index
)

complete_data = (
    operation_data
    .set_index(feature_key)
    .loc[complete_windows]
    .reset_index()
)

print(
    f"Complete windows: "
    f"{len(complete_windows):,}"
)


# ============================================================
# Window metadata
# ============================================================

window_metadata = (
    complete_data[metadata_columns]
    .drop_duplicates(
        subset=feature_key
    )
)


# ============================================================
# Helper function: slope
# ============================================================

def calculate_slope(values):
    """
    Calculate linear slope within one 60-second window.
    """

    x = np.arange(len(values))

    return np.polyfit(
        x,
        values,
        1
    )[0]


# ============================================================
# 1. Process statistics
#
# Includes all 30 process variables.
# ============================================================

process_features = (
    complete_data
    .groupby(feature_key)[feature_variables]
    .agg([
        "mean",
        "std",
        "min",
        "max"
    ])
)

process_features.columns = [
    f"{sensor}_{stat}"
    for sensor, stat
    in process_features.columns
]

process_features = (
    process_features
    .reset_index()
)


# ============================================================
# 2. Window slopes
#
# Includes all 30 process variables.
# ============================================================

window_slopes = (
    complete_data
    .groupby(feature_key)[feature_variables]
    .agg(calculate_slope)
)

window_slopes.columns = [
    f"{sensor}_slope"
    for sensor
    in window_slopes.columns
]

window_slopes = (
    window_slopes
    .reset_index()
)


# ============================================================
# 3. Constant indicators
#
# Applied to actuator/state variables.
#
# We do NOT need constant indicators for continuous
# temperature/process variables because std already
# captures their within-window variability.
# ============================================================

window_constants = (
    complete_data
    .groupby(feature_key)[actuator_variables]
    .std()
    .eq(0)
    .astype("int8")
)

window_constants.columns = [
    f"{sensor}_constant"
    for sensor in window_constants.columns
]

window_constants = (
    window_constants
    .reset_index()
)


# ============================================================
# 4. Temperature correlations
# ============================================================

correlation_rows = []

for (identifier, window), group in (
    complete_data.groupby(feature_key)
):

    correlations = (
        group[temperature_sensors]
        .corr()
    )

    row = {
        "identifier": identifier,
        "window": window,
    }

    for i, sensor_a in enumerate(
        temperature_sensors
    ):

        for sensor_b in temperature_sensors[i + 1:]:

            row[
                f"{sensor_a}_{sensor_b}_corr"
            ] = correlations.loc[
                sensor_a,
                sensor_b
            ]

    correlation_rows.append(row)


window_correlations = pd.DataFrame(
    correlation_rows
)


# ============================================================
# 5. Temperature-pair differences
# ============================================================

difference_columns = []

for sensor_a, sensor_b in difference_pairs:

    column_name = (
        f"{sensor_a}_{sensor_b}_diff"
    )

    complete_data[column_name] = (
        complete_data[sensor_a]
        - complete_data[sensor_b]
    )

    difference_columns.append(
        column_name
    )


# ============================================================
# 5a. Difference statistics
# ============================================================

difference_features = (
    complete_data
    .groupby(feature_key)[difference_columns]
    .agg([
        "mean",
        "std",
        "min",
        "max"
    ])
)

difference_features.columns = [
    f"{column}_{stat}"
    for column, stat
    in difference_features.columns
]

difference_features = (
    difference_features
    .reset_index()
)


# ============================================================
# 5b. Difference slopes
# ============================================================

difference_slopes = (
    complete_data
    .groupby(feature_key)[difference_columns]
    .agg(calculate_slope)
)

difference_slopes.columns = [
    f"{column}_slope"
    for column
    in difference_slopes.columns
]

difference_slopes = (
    difference_slopes
    .reset_index()
)


# ============================================================
# 5c. Combine difference features
# ============================================================

difference_features = (
    difference_features
    .merge(
        difference_slopes,
        on=feature_key,
        how="inner",
        validate="one_to_one"
    )
)


# ============================================================
# 6. Combine all feature blocks
# ============================================================

window_features = (
    process_features

    .merge(
        window_slopes,
        on=feature_key,
        how="inner",
        validate="one_to_one"
    )

    .merge(
        window_constants,
        on=feature_key,
        how="inner",
        validate="one_to_one"
    )

    .merge(
        window_correlations,
        on=feature_key,
        how="inner",
        validate="one_to_one"
    )

    .merge(
        difference_features,
        on=feature_key,
        how="inner",
        validate="one_to_one"
    )
)


# ============================================================
# 7. Add metadata exactly once
# ============================================================

window_features = (
    window_metadata
    .merge(
        window_features,
        on=feature_key,
        how="inner",
        validate="one_to_one"
    )
)


# ============================================================
# Final validation
# ============================================================

print("\nFinal window_features")
print("---------------------")

print(
    "Shape:",
    window_features.shape
)

print(
    "Duplicate windows:",
    window_features
    .duplicated(
        subset=feature_key
    )
    .sum()
)

print(
    "Missing target:",
    window_features[
        "anomaly_label"
    ].isna().sum()
)

print("\nTarget distribution:")

print(
    window_features[
        "anomaly_label"
    ]
    .value_counts()
    .sort_index()
)


# ============================================================
# Save to parquet
# ============================================================


window_features.to_parquet(
    "../data/processed/notebooks/window_features.parquet",
    index=False
)

print(f"Data saved to parquet!")


# ============================================================
# Feature count
# ============================================================

feature_columns = [
    column
    for column in window_features.columns
    if column not in metadata_columns
]

print(
    "\nNumber of ML features:",
    len(feature_columns)
)

Rows: 693,790
Experiments: 119
Windows: 11628
Complete windows: 11,512

Final window_features
---------------------
Shape: (11512, 250)
Duplicate windows: 0
Missing target: 0

Target distribution:
anomaly_label
0.0    8887
1.0     386
2.0    1698
3.0     541
Name: count, dtype: int64
Data saved to parquet!

Number of ML features: 240
